Matsci-rag 实际抽取与 QA 测试

MatSci-RAG Practical Extraction and QA Testing

# 基本配置 Basic Configuration

In [ ]:
from main_pipeline import (
    PipelineConfig,
    build_pipeline_from_output_dir,
)

config = PipelineConfig(
    output_dir=r"examples\test_case\output",

    embedding_model_path=r"D:\BGE_large_en_1.5v",
    rerank_model_path=r"D:\BGE-rerank-large",

    env_file_path=r"C:\Users\12279\ZHIPU.env",
    llm_model_name="GLM-4.1V-Thinking-Flash",

    device="cuda",

    # Manuscript / Table S3 settings
    retrieval_top_k=20,
    rerank_top_n=10,

    max_evidence_objects=20,
    max_textual_evidence_tokens=5120,
    max_visual_tabular_items=5,

    # 开启图像输入
    enable_multimodal=True,

    # 建议测试时先保持 False
    # 如果 LLM 出错，直接报错，而不是用 fallback 假结果
    allow_generation_fallback=False,
)

pipeline, graph = build_pipeline_from_output_dir(config)

print("Pipeline loaded.")
print("doc_id:", graph.doc_id)
print("source_file:", graph.source_file)

Pipeline loaded.
doc_id: test
source_file: test.md


# QA 测试

In [3]:
qa_query = """
What are the main factors affecting the γ ’ phase size reported in this study?
"""

qa_result = pipeline.run(
    query=qa_query,
    graph=graph,
    task_mode="qa",
    retrieval_top_k=20,
    rerank_top_n=10,
)

qa_result

{'task_mode': 'qa',
 'query': '\nWhat are the main factors affecting the γ ’ phase size reported in this study?\n',
 'task_output': "The main factors affecting the γ' phase size reported in this study are the addition of Al, Ti, and Mo, which significantly increase the γ' size. Conversely, the addition of Cr or substituting Nb for Ti can decrease the γ' size. The potency for increasing the γ' size is in the order: Al > Ti > Mo > Nb > Ta. Additionally, the initial γ' size is related to the coarsening rate, with larger initial sizes leading to higher coarsening rates and larger final sizes. The volume fraction of γ' is positively correlated with the γ' size, meaning higher volume fractions correspond to larger sizes. The presence of Mo and W can also influence the γ' size, with Mo showing a higher potency compared to W. Excessive addition of W or Mo can lead to the precipitation of secondary phases, with Mo having a more significant effect. The addition of W can significantly increase th

# 抽取测试

In [5]:
import json

extraction_query = (
    "Extract the alloy composition, processing conditions, "
    "experimental conditions, and reported property."
)

extraction_result = pipeline.run(
    query=extraction_query,
    graph=graph,
    task_mode="extraction",
    retrieval_top_k=20,
    rerank_top_n=10,
)

print(json.dumps(extraction_result, ensure_ascii=False, indent=2))

{
  "task_mode": "extraction",
  "query": "Extract the alloy composition, processing conditions, experimental conditions, and reported property.",
  "task_output": {
    "composition": {
      "A1": {
        "Co": "25.61±0.72 at%",
        "Ni": "48.70±0.76 at%",
        "Al": "13.92±0.58 at%",
        "W": "2.32±0.23 at%",
        "Ti": "1.78±0.19 at%",
        "Ta": "1.92±0.20 at%",
        "Cr": "3.75±0.29 at%",
        "Nb": "0.36±0.09 at%",
        "Mo": "1.59±0.22 at%"
      },
      "A2": {
        "Co": "33.34±0.34 at%",
        "Ni": "40.36±0.31 at%",
        "Al": "11.13±0.23 at%",
        "W": "2.36±0.10 at%",
        "Ti": "4.17±0.14 at%",
        "Ta": "2.65±0.11 at%",
        "Cr": "4.81±0.16 at%",
        "Nb": "1.04±0.07 at%"
      }
    },
    "processing": {},
    "experimental conditions": {
      "temperature": "850 °C",
      "time": "1000 h"
    },
    "property": {
      "γ' solvus temperature": {
        "A1": "1050°C",
        "A2": "1095°C"
      },
      "γ'